**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/chan-2024"], check=True)
    os.chdir("/content/encore/entries/chan-2024")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Reproducing Chan et al. (2024) on Dryad doi:10.5061/dryad.vt4b8gtzk

Paper: Chan, Parasurama, Atlas, Xu, Jongebloed, Alexander, Langenhan, Thornton and Riffell
(2024), "Olfaction in the Anthropocene: NO3 negatively affects floral scent and nocturnal
pollination", *Science* 383, 607-611, doi:10.1126/science.adi0858.

Claims: `claims/chan-2024.yaml`, written from the author manuscript (NIHMS2120560) with the
Supplementary Materials of the version of record, and checked by a second reader.

Dataset: Dryad doi:10.5061/dryad.vt4b8gtzk, version id 289757 (published 2024-04-22). Dryad's
API lists one published version of this dataset, so it is the version the paper cites (the
paper's ref. 40 names the DOI with no version) and the newest; it is pinned for both reasons.
Dryad needs a token to download, so the files were staged before the run in
`/tmp/dryad/vt4b8gtzk/289757/` and are read there in place; the cell below checks every file's
SHA-256 against Dryad's public file list and prints the digests.

The paper's other deposit: The paper's custom code release on Zenodo, record 8411927
(doi:10.5281/zenodo.8411927, ref. 41, the only version of the record), holds data the Dryad
deposit lacks: the raw Vocus traces of Fig. 2D and Table S9, the Fig. 3B wind tunnel sheet, the
fruit-set counts in the Fig. 1E R script, and the GEOS-Chem output of Fig. 4. It is the paper's
own deposit, so claims resting on it are recorded as `data: elsewhere` and the record is listed
under `inputs`. It is streamed from Zenodo (the zip is read by ranges); its MD5 is printed below.
The outside inputs are the land mask of the Supplementary Methods (global-land-mask 1.0.0), the bottom-level
MERRA-2 wind of July and January 2013 from the GEOS-Chem data bucket, and the city coordinates and populations
of the UN World Urbanization Prospects 2018 (see the Fig. 4 section).

No analysis here draws random numbers, so no seeds are used.

| Claim | Tested here | Data |
|---|---|---|
| fig1d-assemblage-day-night | blocked, data missing | |
| fig1e-bagged-fewer-fruit | yes | Zenodo R script counts |
| fig1e-nocturnal-exclusion | yes | Zenodo R script counts |
| fig1e-groups | yes | Zenodo R script counts |
| nocturnal-share-of-fruit | yes | Zenodo counts, Dryad Table S10 |
| fig2a-gcead-monoterpenes | blocked, data missing | |
| table-s6-emission-rates | blocked, data missing | |
| fig2d-no3-degrades-monoterpenes | yes | Zenodo raw Vocus traces (Dryad Fig. 2D file as a variant) |
| fig2d-other-compounds-unchanged | yes | Zenodo raw Vocus traces |
| figs5-blend-oxidation | yes | Dryad benzene-cation CIMS files |
| fig2-dose-dependent | blocked, data missing | |
| figs6-oxidation-products | yes | Dryad FigS6_CIMS_data.xlsx |
| fig3b-hyles-no3-eliminates | yes | Zenodo Fig. 3B sheet |
| fig3b-manduca-no3-halves | yes | Zenodo Fig. 3B sheet |
| fig3b-o3-no2-no-effect | yes | Zenodo Fig. 3B sheet |
| fig3b-proxy-mixture | yes | Zenodo Fig. 3B sheet |
| figs7-manduca-sensitivity | yes | Dryad EAG and dose-response files |
| fig3d-field-no3-fewer-visits | yes | Dryad Table_S10.xlsx |
| fig3d-real-and-synthetic-alike | yes | Dryad Table_S10.xlsx |
| fig3d-visits-per-night | yes | Dryad Table_S10.xlsx |
| fig4ab-no3-shortens-recognition | yes | Zenodo GEOS-Chem, MERRA-2 wind, WUP 2018 cities, land mask |
| table-s13-recognition-distances | yes | Zenodo GEOS-Chem, MERRA-2 wind, WUP 2018 cities |
| fig4c-preindustrial-change | yes | Zenodo GEOS-Chem, WUP 2018 cities, land mask |
| interannual-similar | blocked, data missing | |
| figs8-no3-faster-than-o3 | yes | Zenodo GEOS-Chem, land mask |
| figs9-urban-regressions | yes | Zenodo GEOS-Chem, MERRA-2 wind, WUP 2018 cities |
| table-s1-field-conditions | yes | Dryad field logs |

In [ ]:
import warnings
import resource
import time

import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt

import chan

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
T0 = time.time()
print("Dryad", chan.DRYAD_DOI, "version", chan.DRYAD_VERSION, "| Zenodo", chan.ZENODO_DOI)

## Files of the dataset and of the Zenodo release

In [ ]:
files = chan.check_dryad_files()
print(files.to_string(index=False))
print("files listed:", len(files), "| staged copies matching Dryad's SHA-256:", int(files.matches.sum()))
print("Zenodo record", chan.ZENODO_RECORD, chan.zenodo_md5())
for name, size in chan.zenodo_listing():
    print(f"  {size:>9}  {name}")

### Everything the Dryad files hold

Every column of every table, and every sheet of every workbook (protocol, author, step 1.1).

In [ ]:
D = chan.DRYAD_DIR
for f in sorted(D.iterdir()):
    if f.suffix == ".xlsx":
        for sheet, df in pd.read_excel(f, sheet_name=None).items():
            print(f"{f.name} [sheet {sheet}] shape {df.shape}: {list(df.columns)}")
    elif f.name.startswith("Fig2D"):
        df = chan.fig2d_tof(); print(f"{f.name} shape {df.shape}: {list(df.columns)}")
    elif f.name.startswith("Fig_S4_") and f.suffix == ".txt":
        print(f"{f.name}: 2 unnamed columns (time s, ion counts), {len(np.loadtxt(f))} rows")
    elif f.suffix == ".atf":
        hdr = f.read_text(errors="replace").splitlines()[:10]
        print(f"{f.name}: Axon text file, columns {hdr[9]}; {sum(1 for _ in open(f)) - 10} rows")
    elif f.name.startswith("FigS3_O3"):
        print(f"{f.name}: columns {list(pd.read_csv(f, nrows=2).columns)}")
    elif f.name.startswith("FigS3_2018"):
        print(f"{f.name}: columns {[l for l in f.read_text(encoding='latin-1').splitlines() if l.startswith('\"Date')]}")
    elif f.name.startswith("TableS1_FigS3_CAPS"):
        print(f"{f.name}: columns {list(pd.read_csv(f, comment='%', nrows=1, skipinitialspace=True).columns)}")
    elif f.name.startswith("Fig_S5_Manduca"):
        print(f"{f.name}: columns {list(pd.read_csv(f).columns)}")
    elif f.name.startswith("TableS8"):
        print(f"{f.name}: first lines {f.read_text(errors='replace').splitlines()[:2]}")
    elif f.name.startswith("Table_S1_anemometer"):
        print(f"{f.name}: header {chan.anemometer_header()!r}, then 4 unnamed columns u v w T")
    else:
        print(f"{f.name}: {f.stat().st_size} bytes (documentation or code)")

## Data check

What was loaded, with counts and the ranges and units of every variable used, checked against
the README and the paper.

In [ ]:
o3 = chan.load_o3(); caps = chan.load_caps(); ib = chan.load_ibutton(); anem = chan.load_anemometer()
s10_sheet, s10 = chan.table_s10()
fruit, fruit_line = chan.fruit_set_counts()
fig3b = chan.wind_tunnel_fig3b(); s7c = chan.wind_tunnel_s7c()
eag_raw, eag = chan.eag()
gc = chan.geos_chem()
print("Field ozone (ppb):"); print(o3.groupby("file").agg(n=("O3", "size"), start=("time", "min"), end=("time", "max"), min=("O3", "min"), max=("O3", "max")))
print("\nCAPS NO2 (ppb):"); print(caps.groupby("file").agg(n=("NO2", "size"), start=("time", "min"), end=("time", "max"), finite=("NO2", "count"), min=("NO2", "min"), max=("NO2", "max")))
print("CAPS status codes:", caps.Status.value_counts().to_dict())
print("\niButtons (deg C, RH %):"); print(ib.groupby("file").agg(n=("RH", "size"), start=("time", "min"), end=("time", "max"), tmin=("temp_C", "min"), tmax=("temp_C", "max"), rhmin=("RH", "min"), rhmax=("RH", "max")))
print("iButton temperature, midnight to 6 am, mean:", round(ib[chan.night_mask(ib.time)].temp_C.mean(), 2), "C (Table S1: 20.65)")
print("iButton sampling interval (median):", ib.groupby("file").time.apply(lambda s: s.diff().median()).iloc[0])
print("\nAnemometer:", len(anem), "rows; at 32 Hz that is", round(len(anem) / 32 / 3600, 2), "h")
print(anem.describe().loc[["mean", "min", "max"]].round(2), "(u, v, w in cm/s, T in K)")
print("\nTable S10 sheet", s10_sheet); print(s10)
print("\nFruit-set counts from the Zenodo R script:", fruit_line.strip()); print(fruit)
for k, v in fig3b.items(): print("\nFig. 3B sheet", k); print(v.dropna(axis=1, how="all"))
for k, v in s7c.items(): print("\nFig. S7C (file 'Fig_S5C') sheet", k); print(v)
print("\nEAG values per species and concentration (mV):")
print(eag.groupby(["species", "conc"]).mV.agg(["count", "mean", "min", "max"]).round(2))
print("\nGEOS-Chem 12.1.0 arrays:", {k: v.shape for k, v in gc.items()})
print("NO3 v/v range", gc["Conc_NO3_Y0"].min(), gc["Conc_NO3_Y0"].max(), "| O3 v/v range", gc["Conc_O3_Y0"].min(), gc["Conc_O3_Y0"].max())
print("pressure (hPa)", gc["Pres_Y0"].min().round(1), gc["Pres_Y0"].max().round(1), "| temperature (K)", gc["Temp_Y0"].min().round(1), gc["Temp_Y0"].max().round(1))
print("lat", gc["lat"].min(), gc["lat"].max(), "| lon", gc["lon"].min(), gc["lon"].max(), "| months (Jan 2013 to Feb 2014):", gc["Conc_NO3_Y0"].shape[-1])

Checks of the description against the values:

- Ozone is in ppb (field values 10 to 48, one excursion to 137), with dates and times as the
  README says; the three logs cover the nights of 26-27 July, 2 August and 11 August 2018.
- CAPS NO2 is in ppb; rows during zeroing (status 11004 and 12004) carry NaN, and status 4 rows
  include one value of -437 ppb. The status codes are not described.
- The iButtons log every 3 min, not at the 0.5 Hz the Supplementary Methods state, over 11 to 16
  July, 25 to 30 July and 1 to 5 August 2018. Their date column is an Excel serial date.
- The anemometer file has one header (a PuTTY log started 2018-08-01 22:15:22) and 1,387,308 rows,
  12.0 h at 32 Hz; no timestamps.
- All field files are from 2018; Table S1 says 2018 and 2019.
- GEOS-Chem mixing ratios are v/v (NO3 up to 46 ppt, O3 4 to 86 ppb), pressure in hPa,
  temperature in K, on the 2 x 2.5 degree grid with 14 monthly means. The file holds no wind.
- The flowtube and field streams are separate experiments with no shared trial table; the only
  clock questions are the stage timing within each flowtube run (see the claims) and the field
  logs, whose dates overlap on 26-27 July and 1-2 August.

### Problems found in the data

In [ ]:
# The Hyles column repeats eight consecutive EAG values (1e-5 to 3 floral equivalents) of one
# dose series in the next series (file rows 12-19 and 21-28).
raw, _ = chan.eag()
a, b = raw.loc[10:17, ["conc_hyles", "hyles"]].reset_index(drop=True), raw.loc[19:26, ["conc_hyles", "hyles"]].reset_index(drop=True)
print(pd.concat([a.add_suffix(" rows 12-19"), b.add_suffix(" rows 21-28")], axis=1))
print("identical:", bool((a.values == b.values).all()))
print("the 1e-6 value before each series, file rows 11 and 20 (conc, mV):", raw.loc[[9, 18], ["conc_hyles", "hyles"]].values.tolist())
vals = raw.hyles.values
runs = [(i + 2, j + 2) for i in range(len(vals) - 7) for j in range(i + 1, len(vals) - 7)
        if np.array_equal(vals[i:i + 8], vals[j:j + 8])]
print("runs of 8 identical Hyles values:", runs, "| in the Manduca column:",
      [(i, j) for i in range(len(raw) - 7) for j in range(i + 1, len(raw) - 7) if np.array_equal(raw.manduca.values[i:i + 8], raw.manduca.values[j:j + 8])])

In [ ]:
# The EAG file holds 159 values per species, 16 to 18 per concentration over 9 concentrations
# (1e-6 to 3), where the Methods state 159 moths per species and 8 concentrations
# (1:100,000,000 to 1:33).
raw, eag = chan.eag()
print(eag.groupby("species").size().to_dict())
print(eag.groupby(["species", "conc"]).size().unstack(0))

In [ ]:
# Fig. 3B's Manduca "odor blend" (Methods: 1/100 floral equivalent) is 11 of 12 moths, the same
# numbers as the 1x row of the Fig. S7C dose-response file; the 0.01x row is 10 of 14.
f3 = chan.wind_tunnel_fig3b()["Sheet1"]; s7 = chan.wind_tunnel_s7c()["Sheet1"]
print(f3[f3.Species == "M. sexta"].iloc[:1, :4].to_string(index=False))
print(s7.to_string(index=False))
# Moths per stimulus and per species, against the Methods ("138 M. sexta and 96 H. lineata"; "For each
# odor stimulus, 15-42 moths were tested") and the Fig. 3B legend ("n = 15 to 35 moths per treatment").
f3n = f3.dropna(subset=["Moths tested"])
print("moths per species:", f3n.groupby("Species")["Moths tested"].sum().astype(int).to_dict(), "(Methods: M. sexta 138, H. lineata 96)")
print("moths per stimulus:", sorted(f3n["Moths tested"].astype(int).tolist()), "(Methods: 15 to 42)")


In [ ]:
# Fig2D_ToFdata.txt: the README gives time in minutes; the step is 1.0035, as in the raw Vocus
# files where it is seconds. Its b-pinene trace does not give Table S9's changes under the
# notebook's windows, while its ocimene trace does.
d = chan.fig2d_tof()
print("time step:", np.round(np.diff(d.Time.dropna())[:3], 4), "| last time:", round(d.Time.max(), 1))
for comp, col in [("b-pinene", "beta-pinene"), ("ocimene", "beta-ocimene")]:
    c = chan.VOCUS_CODE[comp]; x = d[col].dropna().to_numpy(); _, y = chan.vocus_trace(comp)
    base = x[slice(*c["clean"])].mean(); base_raw = y[slice(*c["clean"])].mean()
    print(comp, "| Dryad file: O3 %.1f%%, NO3 %.1f%%" % tuple(100 * (x[slice(*c[k])].mean() / base - 1) for k in ("o3", "no3")),
          "| raw Zenodo trace: O3 %.1f%%, NO3 %.1f%%" % tuple(100 * (y[slice(*c[k])].mean() / base_raw - 1) for k in ("o3", "no3")),
          "| correlation %.3f" % np.corrcoef(x[:len(y)], y[:len(x)])[0, 1])

In [ ]:
# CAPS files: the IgorTime column is not the time of the Timestamp column (its offset differs by
# file), and status codes are undocumented; status-4 rows include NO2 = -437 ppb.
caps = chan.load_caps()
igor_epoch = (caps.time - pd.Timestamp("1904-01-01")).dt.total_seconds()
print((caps.IgorTime - igor_epoch).groupby(caps.file).median().div(86400).round(1).rename("IgorTime minus Timestamp (days)").to_string())
print(caps.groupby("Status").NO2.agg(["size", "count", "min", "max"]))

In [ ]:
# The anemometer file has no timestamps and a single log header; the README says the data are
# concatenated over the sampling times, so the midnight to 6 am rows cannot be located.
print(chan.anemometer_header())
anem = chan.load_anemometer()
print(len(anem), "rows =", round(len(anem) / 32 / 3600, 2), "h at 32 Hz; largest temperature steps (K):",
      np.round(np.sort(np.abs(np.diff(anem["T"].to_numpy())))[-3:], 2))

In [ ]:
# What the Dryad deposit holds, against the paper's Zenodo release: Zenodo files with an identical
# Dryad file (by SHA-256), and those without one. Neither holds the pollinator visit log (Table
# S2), the GC-MS samples behind Table S6, more than one GC-EAD antenna per species, or wind fields.
import hashlib
fl = chan.check_dryad_files()
dryad_sha = dict(zip(fl.sha256, fl.file))
same, only_zenodo = [], []
for name, size in chan.zenodo_listing():
    h = hashlib.sha256(chan.zread(name)).hexdigest()
    (same if h in dryad_sha else only_zenodo).append(name if h not in dryad_sha else f"{name} = {dryad_sha[h]}")
print(len(same), "Zenodo files identical to a Dryad file, e.g.", same[:2])
print(len(only_zenodo), "Zenodo files with no identical Dryad file:"); print("\n".join("  " + n for n in only_zenodo))
print("Dryad GC-EAD traces:", [n for n in fl.file if n.endswith(".atf")])
print("Dryad files whose name mentions GEOS, S2 visits, S3 fruit or Fig 3:", [n for n in fl.file if any(k in n for k in ("GEOS", "Table_S2", "Table_S3", "Fig3", "Fig_3"))])

In [ ]:
# FigS6_CIMS_data.xlsx holds two time segments with no label saying which oxidant each is.
s6 = pd.read_excel(chan.dryad("FigS6_CIMS_data.xlsx"))
s6["time"] = pd.to_datetime(dict(year=s6.yyyy, month=s6.mo, day=s6.dd, hour=s6.hh, minute=s6.mm, second=s6.ss))
gap = s6.time.diff() > pd.Timedelta("10min")
print(s6.groupby(gap.cumsum()).time.agg(["min", "max", "size"]))
print("columns:", list(s6.columns))

## Fig. 1E: fruit set under pollinator exclusion

> fig1e-bagged-fewer-fruit: "Bagged flowers (all pollinators excluded) set fewer fruit than
> unbagged flowers (0.114 against 0.794)."
>
> fig1e-nocturnal-exclusion: "Excluding nocturnal pollinators lowers fruit set below that of
> unbagged controls (0.477 against 0.794)."
>
> fig1e-groups: "Crossed, diurnal-exclusion and unbagged flowers do not differ in fruit set;
> nocturnal exclusion does not differ from crossed or diurnal exclusion; total exclusion differs
> from all (letters a, a,b, c, b, a,b)."

The counts are read from the `pairwise.prop.test` call of the paper's R script (Zenodo); the
Dryad deposit has no fruit-set data. The test is R's `pairwise.prop.test` with its defaults
(Yates correction, Holm adjustment), as the criteria say, implemented in `chan.pairwise_prop_test`.
The letters are those of Fig. 1E and the R script (`SigLabels`).

In [ ]:
fruit, _ = chan.fruit_set_counts()
fruit["proportion"] = fruit.fruit / fruit.n
fruit["sd (binomial)"] = np.sqrt(fruit.proportion * (1 - fruit.proportion) / fruit.n)
print(fruit.round(4).to_string(index=False))
pw = chan.pairwise_prop_test(list(fruit.treatment), list(fruit.fruit), list(fruit.n))
table_s4 = {("total exclusion", "crossed"): 3.2e-6, ("total exclusion", "nocturnal exclusion"): 0.0032,
            ("total exclusion", "unbagged"): 1.7e-10, ("total exclusion", "diurnal exclusion"): 4.8e-7,
            ("crossed", "nocturnal exclusion"): 0.4645, ("crossed", "unbagged"): 0.5269,
            ("crossed", "diurnal exclusion"): 1.0, ("nocturnal exclusion", "unbagged"): 0.0082,
            ("nocturnal exclusion", "diurnal exclusion"): 0.4645, ("unbagged", "diurnal exclusion"): 0.4645}
letters = {"unbagged": "a", "crossed": "ab", "total exclusion": "c", "nocturnal exclusion": "b", "diurnal exclusion": "ab"}
pw["Table S4"] = [table_s4[(a, b)] for a, b in zip(pw.a, pw.b)]
pw["share a letter"] = [bool(set(letters[a]) & set(letters[b])) for a, b in zip(pw.a, pw.b)]
pw["consistent with letters"] = np.where(pw["share a letter"], pw.p_holm >= 0.05, pw.p_holm < 0.05)
print(pw.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
p_tot = pw.query("a == 'total exclusion' and b == 'unbagged'").p_holm.item()
p_noc = pw.query("a == 'nocturnal exclusion' and b == 'unbagged'").p_holm.item()
print(f"\nunbagged {fruit.proportion[3]:.3f} (expected 0.794 +- 0.051), total exclusion {fruit.proportion[0]:.3f} (0.114 +- 0.0479), Holm p {p_tot:.3g}")
print(f"nocturnal exclusion {fruit.proportion[2]:.3f} (0.477 +- 0.0753), Holm p against unbagged {p_noc:.4f} (0.0082 +- 0.00082)")
print("pairs consistent with the letters:", int(pw["consistent with letters"].sum()), "of", len(pw))
noyates = chan.pairwise_prop_test(list(fruit.treatment), list(fruit.fruit), list(fruit.n), correct=False)
print("variant without Yates correction, Holm p:", dict(zip(noyates.a + " / " + noyates.b, noyates.p_holm.round(4))))

**Result: fig1e-bagged-fewer-fruit, fig1e-nocturnal-exclusion and fig1e-groups are reproduced.** The counts in the paper's R script give the Table S3 proportions (unbagged 0.794, total exclusion 0.114, nocturnal exclusion 0.477), and R's pairwise.prop.test gives Holm-adjusted p of 1.71e-10 for total exclusion against unbagged and 0.0082 for nocturnal exclusion against unbagged. All ten adjusted p equal Table S4 at its printed precision, and all ten pairs agree with the letters of Fig. 1E. These are the paper's own counts; the flower-level data are not deposited, so this checks the paper's computation on its published counts.

## Share of fruit set due to night pollinators

> nocturnal-share-of-fruit: "Hawkmoths and crepuscular bees account for about 40% of fruit set,
> so the 70% drop in visitation under NO3 would cut total fruit set by about 28%."

Readings of the criterion: (1) share = (unbagged - nocturnal exclusion) / unbagged; (2) share =
diurnal exclusion / unbagged. The reduction is the share times the drop in visitation of
fig3d-visits-per-night (both of its readings of the untreated flower are shown).

In [ ]:
fruit, _ = chan.fruit_set_counts()
p = dict(zip(fruit.treatment, fruit.fruit / fruit.n))
_, s10 = chan.table_s10()
night = s10.loc["Visits/hour"].astype(float) * 7
drops = {"live flower": 1 - night["NO3 oxid. floral odor"] / night["Live flower"],
         "floral odor": 1 - night["NO3 oxid. floral odor"] / night["Floral odor"]}
shares = {"(1) (unbagged - nocturnal excl.) / unbagged": (p["unbagged"] - p["nocturnal exclusion"]) / p["unbagged"],
          "(2) diurnal excl. / unbagged": p["diurnal exclusion"] / p["unbagged"]}
rows = []
for sname, share in shares.items():
    for dname, drop in drops.items():
        rows.append({"share reading": sname, "untreated flower": dname, "share %": 100 * share, "drop %": 100 * drop,
                     "reduction %": 100 * share * drop,
                     "share within 40 +- 10": abs(100 * share - 40) <= 10, "reduction within 28 +- 11": abs(100 * share * drop - 28) <= 11})
print(pd.DataFrame(rows).round(2).to_string(index=False))

**Result: reproduced under the criterion's first reading.** With the share read as (unbagged minus nocturnal exclusion) over unbagged, the night pollinators account for 39.9% of fruit set and the 70.6% drop in visitation (live flower) gives a 28.1% reduction (28.5% with the synthetic floral odor as the untreated flower). The second reading (diurnal exclusion over unbagged, 82.8%) gives 58.4% and is outside both tolerances.

## Fig. 2D and Table S9: flowtube oxidation of single compounds

> fig2d-no3-degrades-monoterpenes: "In the flowtube, O3 lowers b-pinene and cis-b-ocimene by
> about 30% (33% and 30%), while NO3 lowers them by 84% and 67%."
>
> fig2d-other-compounds-unchanged: "Other scent compounds (2-methylbutanal oxime, benzaldehyde,
> nonanal) show little or no change with either oxidant."

The raw Vocus traces are in the Zenodo release only. Each trace is processed as the Dryad notebook
`Fig_S4_Table_S9_VOCUS_analysis_and_plotting.ipynb` does (its cells 29 to 33: section, linear
detrend for the aldoxime and nonanal, division by the baseline window mean; `chan.VOCUS_CODE`).
That notebook, in the dataset the paper cites for its data (ref. 40), also gives the clean, O3
and NO3 index windows of each compound and computes Table S9 from them. A reader answered the
question this attempt asked (`questions/chan-2024.yaml`) and amended the criteria: the notebook's
windows are the paper's method and decide the outcome of both claims, with NO3 against clean air
as the notebook computes it; the readings (a) last half of each stage and (b) whole stage are
shown as variants. Stage onsets for the monoterpenes are placed where the smoothed trace crosses half-way
between stage levels; the other compounds show no step, and their stage extents are the
notebook's windows. NO3 readings: (i) against clean air, (ii) against the O3 stage. The test is
Table S9's (`chan.table_s9_test`).

In [ ]:
# The paper's Table S9 code: the functions and cells 29 to 33 of the Dryad notebook
# Fig_S4_Table_S9_VOCUS_analysis_and_plotting.ipynb, printed beside chan.VOCUS_CODE, which transcribes their
# sections, detrend, baseline windows and comparison windows. chan.vocus_trace detrends as the notebook's
# `detrend` does, m * (time - start), with time in seconds and start an index, as written there.
import json
dn = json.loads(chan.dryad("Fig_S4_Table_S9_VOCUS_analysis_and_plotting.ipynb").read_text())
for i in [1, 2, 12, 17, 29, 30, 31, 32, 33]:
    print(f"--- Dryad notebook cell {i}:"); print("".join(dn["cells"][i]["source"]).strip())
print("\n--- chan.VOCUS_CODE:")
for k, v in chan.VOCUS_CODE.items(): print(" ", k, v)
# The notebook averages the aldoxime's clean air over 0:300 and tests 100:200; the test on 0:300 as a variant.
t_, y_ = chan.vocus_trace("aldoxime"); c_ = chan.VOCUS_CODE["aldoxime"]
for st in ["o3", "no3"]:
    for cw in [(100, 200), (0, 300)]:
        p_, test_ = chan.table_s9_test(y_[slice(*cw)], y_[slice(*c_[st])])
        print(f"aldoxime {st.upper()} against clean air {cw}: change {100 * (y_[slice(*c_[st])].mean() / y_[slice(*c_['clean'])].mean() - 1):.2f}%, p = {p_:.3g} ({test_})")


In [ ]:
ox = pd.concat([chan.oxidation_table(c) for c in ["b-pinene", "ocimene", "aldoxime", "benzaldehyde", "nonanal"]], ignore_index=True)
print(ox.to_string(index=False, float_format=lambda v: f"{v:.4g}"))

In [ ]:
exp = {("b-pinene", "O3"): (-33, 3.3), ("b-pinene", "NO3"): (-84, 8.4), ("ocimene", "O3"): (-30, 3), ("ocimene", "NO3"): (-67, 6.7)}
rows = []
for reading in ["code windows", "a (last half of stage)", "b (whole stage)"]:
    for no3_ref in ["(i) vs clean", "(ii) vs O3"]:
        vals, ok, ps = [], [], []
        for comp in ["b-pinene", "ocimene"]:
            r = ox[(ox.compound == comp) & (ox.reading == reading)].iloc[0]
            o3v, no3v = r["O3 vs clean %"], (r["NO3 vs clean %"] if no3_ref.startswith("(i)") else r["NO3 vs O3 %"])
            pno3 = r["p NO3"] if no3_ref.startswith("(i)") else r["p NO3 vs O3"]
            for k, v in [("O3", o3v), ("NO3", no3v)]:
                e, tol = exp[(comp, k)]; vals.append(round(v, 1)); ok.append(abs(v - e) <= tol)
            ps += [r["p O3"], pno3]
        rows.append({"windows": reading + (" (decides)" if reading == "code windows" and no3_ref.startswith("(i)") else ""), "NO3 reading": no3_ref, "b-pinene O3, NO3, ocimene O3, NO3 (%)": vals,
                     "all within tolerance": all(ok), "max p": max(ps), "all p < 0.001": max(ps) < 0.001})
print(pd.DataFrame(rows).to_string(index=False))
# The notebook states each change as the clean-air mean minus the stage mean of the normalized
# trace (cells 29 and 30), a difference in units of the baseline window; the table above gives
# the ratio to the clean-air mean. Both forms under the notebook's windows:
for comp in ["b-pinene", "ocimene"]:
    c = chan.VOCUS_CODE[comp]; t_, y_ = chan.vocus_trace(comp)
    m = {k: y_[slice(*c[k])].mean() for k in ("clean", "o3", "no3")}
    print(f"{comp}: difference form O3 {100 * (m['o3'] - m['clean']):.2f}, NO3 {100 * (m['no3'] - m['clean']):.2f}; "
          f"ratio form O3 {100 * (m['o3'] / m['clean'] - 1):.2f}, NO3 {100 * (m['no3'] / m['clean'] - 1):.2f}")


**Result: reproduced.** Under the windows of the paper's Table S9 notebook, which the amended criterion takes as the paper's method, O3 lowers b-pinene by 32.5% (Welch t, p = 2.2e-120) and ocimene by 29.8% (Mann-Whitney U, p = 5.0e-54), and NO3 against clean air lowers them by 83.9% (Mann-Whitney U, p = 4.8e-67) and 66.8% (Welch t, p = 2.2e-171): all four within tolerance, each p below 0.001, and the tests and p values those of Table S9. In the notebook's own form, the difference of normalized means, ocimene's changes are 29.9% and 67.0%, also within tolerance. The variants agree: reading (a), the last half of each stage, gives 33.7%, 85.0%, 30.7% and 66.4%, and reading (b), the whole stages, 33.1%, 82.4%, 29.2% and 64.6%, all within tolerance with every p below 1e-52. Against the O3 stage (reading ii) the NO3 changes are 76.1% and 52.7% under the notebook's windows, and ocimene is then outside tolerance. The deposited Fig. 2D file gives 28.9% and 74.4% for b-pinene under the paper's windows (finding).


In [ ]:
other = ox[ox.compound.isin(["aldoxime", "benzaldehyde", "nonanal"])]
cols = ["O3 vs clean %", "p O3", "NO3 vs clean %", "p NO3", "NO3 vs O3 %", "p NO3 vs O3"]
for reading in ["code windows", "a (last half of stage)", "b (whole stage)"]:
    sub = other[other.reading == reading]
    for no3_ref, pcols in [("(i) vs clean", ["p O3", "p NO3"]), ("(ii) vs O3", ["p O3", "p NO3 vs O3"])]:
        pmin = sub[pcols].min().min()
        below = [(c, k) for c in sub.compound for k in pcols if sub.loc[sub.compound == c, k].item() < 0.05]
        decides = " (decides)" if reading == "code windows" and no3_ref.startswith("(i)") else ""
        print(f"{reading + decides:24s} {no3_ref:13s} smallest p {pmin:.3g}; p < 0.05: {below}")
print("\nTable S9 changes (O3, NO3): aldoxime 0, -1; benzaldehyde 0, +2; nonanal 0, -1. Fig. 2D inset benzaldehyde -9, -12.")
print(other[["compound", "reading"] + cols].round(3).to_string(index=False))

**Result: reproduced.** Under the windows of the paper's Table S9 notebook, which the amended criterion makes decisive, with NO3 against clean air as the notebook computes it, no p for the three compounds is below 0.05 (smallest 0.065, aldoxime with O3). The changes are -0.10% and -0.57% for the aldoxime, +0.36% and +2.1% for benzaldehyde, and -0.36% and -0.87% for nonanal, Table S9's values. The variants, which do not decide the outcome, differ: under reading (a) the aldoxime falls by 0.56% with O3 (p = 0.00093) and 0.70% with NO3 (p = 0.00017), and nonanal rises by 4.0% with NO3 (p = 0.025); under reading (b) the aldoxime falls by 0.57% with NO3 (p = 5.2e-05). These effects are below 1% for the aldoxime, and they are significant because each window holds hundreds of time points. NO3 against the O3 stage, which the criterion does not use, gives p = 2.7e-05 for the aldoxime (-0.47%) under the notebook's windows. Benzaldehyde has no p below 0.05 under any reading (changes -2.3% to +2.1%); Table S9 (0% and +2%) and not the Fig. 2D inset (-9% and -12%) matches its data.


## Fig. S5: oxidation of the whole blend (benzene-cation CIMS)

> figs5-blend-oxidation: "When the whole floral blend is oxidized with NO3, monoterpene signals
> drop while 2-methylbutanal oxime and eucalyptol do not change."

The Dryad files hold the monoterpene, aldoxime and eucalyptol ion counts of this run, at 1 s. The
Dryad notebook (cell 23) plots them from 420 s to the end, normalized by 420 to 520 s, so 420 s is
taken as the start of the clean-air stage. The NO3 onset is placed where the smoothed monoterpene
trace crosses half-way between its clean-air level (420 to 520 s) and its level over the last
100 s. Readings: (a) last half and (b) whole of each stage, with Table S9's test. The criterion, as the
reader amended it, keeps these readings because the Table S9 notebook's index windows belong to
the single-compound Vocus runs and do not transfer to this run. The reagent-ion
file the notebook divides by is not deposited, so the signals are not normalized to it (a
deviation); the notebook's linear detrend of the aldoxime over the first 700 s of the section is
shown as a variant.

In [ ]:
tr = {n: chan.blend_cims(n) for n in ["monoterpene", "aldoxime", "eucalyptol"]}
t, m = tr["monoterpene"]
start = 420
clean_lvl, end_lvl = m[start:start + 100].mean(), m[-100:].mean()
onset = chan.step_onset(m, clean_lvl, end_lvl, start + 100, len(m) - 100)
print("NO3 onset at", t[onset].round(1), "s (row", onset, ")")
wins = {"a (last half)": {"clean": ((start + onset) // 2, onset), "no3": ((onset + len(m)) // 2, len(m))},
        "b (whole)": {"clean": (start, onset), "no3": (onset, len(m))}}
# Variants on drift, none of which decides the outcome: "detrended 0-700" is the linear detrend the paper's
# notebook applies to the aldoxime of this run (cell 23, detrendn(df1, 0, 700) on the section from 420 s),
# applied here to each ion; "detrended clean" fits the line on the clean-air window only and extends it.
# "p 30 s blocks" is a Welch t-test on means of consecutive 30 s blocks, as the 1 s points are autocorrelated
# (lag-1 r of the clean-air window shown).
from scipy import stats
rows = []
for reading, w in wins.items():
    for name, (tt, y) in tr.items():
        for variant in ["raw", "detrended 0-700", "detrended clean"]:
            yy = y.astype(float).copy()
            if variant == "detrended 0-700":
                sec = yy[start:]; k, _ = np.polyfit(np.arange(700), sec[:700], 1); yy[start:] = sec - k * np.arange(len(sec))
            elif variant == "detrended clean":
                a_, b_ = w["clean"]; k, _ = np.polyfit(np.arange(a_, b_), yy[a_:b_], 1); yy = yy - k * (np.arange(len(yy)) - a_)
            c, n3 = yy[slice(*w["clean"])], yy[slice(*w["no3"])]
            p, test = chan.table_s9_test(c, n3)
            cb, nb_ = (x[: len(x) // 30 * 30].reshape(-1, 30).mean(1) for x in (c, n3))
            rows.append({"reading": reading, "ion": name, "variant": variant, "clean": w["clean"], "no3": w["no3"],
                         "change %": 100 * (n3.mean() / c.mean() - 1), "p": p, "test": test, "lag-1 r clean": np.corrcoef(c[:-1], c[1:])[0, 1],
                         "blocks": (len(cb), len(nb_)), "p 30 s blocks": stats.ttest_ind(cb, nb_, equal_var=False).pvalue})
s5 = pd.DataFrame(rows); print(s5.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
fig, ax = plt.subplots(1, 3, figsize=(12, 3))
for a_, (name, (tt, y)) in zip(ax, tr.items()):
    a_.plot(tt, y, lw=0.5); a_.axvline(t[onset], color="r"); a_.axvline(start, color="k"); a_.set_title(name); a_.set_xlabel("s")
plt.tight_layout(); plt.show()
print("plotted ranges:", {k: (round(float(v[1].min()), 1), round(float(v[1].max()), 1)) for k, v in tr.items()})
# The paper's notebook divides each ion of this run by the reagent ion (C6D6) before normalizing (cell 23,
# normalizeri); the reagent-ion trace is not deposited:
print("Dryad files of this run:", sorted(p_.name for p_ in chan.DRYAD_DIR.glob("Fig_S4_*")))
dn = json.loads(chan.dryad("Fig_S4_Table_S9_VOCUS_analysis_and_plotting.ipynb").read_text())
print("".join(dn["cells"][23]["source"]).strip())


**Result: partially reproduced.** After the NO3 onset at 763 s the monoterpene ion falls by 34.1% (reading a) or 32.9% (reading b), p < 1e-63, so the drop holds. The "no change" half fails: under the criterion's test on the raw traces the eucalyptol ion rises by 9.2% and 7.2% (p < 1e-74) and the aldoxime ion falls by 22.2% and 23.9% (p < 1e-52). These changes may be drift: with the notebook's linear detrend over the first 700 s the aldoxime change is +2.4% (p = 0.027, reading a) and -1.2% (p = 0.14, reading b) and the eucalyptol change +3.5% and +1.9%, but the same detrend also takes the monoterpene drop to -3.7% and -4.3%, so it cannot separate drift from chemistry here. On 30 s block means, which allow for the autocorrelation of the 1 s points, the raw eucalyptol rise keeps p < 1e-4 under both readings. The paper's notebook divides these ions by the reagent ion, which is not deposited, so the outcome may reflect that missing trace.

## Fig. S6: oxidation products

> figs6-oxidation-products: "NO3 oxidation of the scent yields organic nitrates (C10H15NO6,
> C10H17NO6, C10H17NO7) as well as C10H16O4 and C10H18O4; O3 yields only the latter two."

The workbook holds two segments without labels (finding above). The Supplementary Methods (Characterization of free-radical oxidation) run each flowtube experiment in three stages, clean air, then O3 ("stage II"), then NO3 ("Stage III was initiated by injecting NO3"), so the earlier segment (05:12 to 05:38) is taken as the O3 stage and the later (06:36 to 06:56) as the NO3 stage. Each segment has its own clean-air rows, so the stage order alone does not settle which run is which; that organic nitrates need NO3 supports the same assignment. In each segment the oxidant onset is the first row where C10H16O4 exceeds twice the median of the segment's first five rows (a standard choice); rows before it are the clean-air level. What counts as a rise above the clean-air level is not defined; three readings are run: a one-sided Welch test (r1, 5 against 9 and 4 against 7 points), every point after the onset above the clean-air mean plus 3 SD (r2), and a mean after the onset at least twice the clean-air mean (r3, a threshold chosen here, not the paper's).


In [ ]:
s6 = pd.read_excel(chan.dryad("FigS6_CIMS_data.xlsx"))
s6["time"] = pd.to_datetime(dict(year=s6.yyyy, month=s6.mo, day=s6.dd, hour=s6.hh, minute=s6.mm, second=s6.ss))
s6["segment"] = (s6.time.diff() > pd.Timedelta("10min")).cumsum().map({0: "O3 (first segment)", 1: "NO3 (second segment)"})
ions = ["C10H16O4", "C10H18O4", "C10H15NO6", "C10H17NO6", "C10H17NO7"]
from scipy import stats
rows = []
for seg, d in s6.groupby("segment", sort=False):
    d = d.reset_index(drop=True)
    onset = int(np.nonzero(d.C10H16O4.to_numpy() > 2 * d.C10H16O4.iloc[:5].median())[0][0])
    for ion in ions:
        pre, post = d[ion].iloc[:onset], d[ion].iloc[onset:]
        p = stats.ttest_ind(post, pre, equal_var=False, alternative="greater").pvalue
        rows.append({"segment": seg, "onset": d.time[onset].strftime("%H:%M"), "n pre": len(pre), "n post": len(post), "ion": ion,
                     "pre mean": pre.mean(), "post mean": post.mean(), "ratio": post.mean() / pre.mean(),
                     "r1 p (Welch, one-sided)": p, "r2 all > mean+3SD": bool((post > pre.mean() + 3 * pre.std()).all()),
                     "r3 ratio >= 2": post.mean() >= 2 * pre.mean()})
s6r = pd.DataFrame(rows); print(s6r.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
for r in ["r1", "r2", "r3"]:
    rise = {(x.segment, x.ion): (x["r1 p (Welch, one-sided)"] < 0.05 if r == "r1" else x[[c for c in s6r.columns if c.startswith(r)][0]]) for _, x in s6r.iterrows()}
    nitr_no3 = all(rise[("NO3 (second segment)", i)] for i in ions[2:]); nitr_o3 = any(rise[("O3 (first segment)", i)] for i in ions[2:])
    c10 = all(rise[(s, i)] for s in ["O3 (first segment)", "NO3 (second segment)"] for i in ions[:2])
    print(f"{r}: nitrates rise with NO3 {nitr_no3}; any nitrate rises with O3 {nitr_o3}; both C10 ions rise in both {c10}; criterion met {nitr_no3 and not nitr_o3 and c10}")
print("README on this file:", [l for l in chan.dryad("README.md").read_text().splitlines() if "S6" in l])


**Result: partially reproduced, resting on two assumptions: that the earlier segment is the O3 stage, as the Supplementary Methods order the stages, and that a rise means at least a doubling (r3), a threshold this attempt chose.** In the later segment, taken as NO3, all five ions rise (nitrates 17 to 23 times their clean-air level, C10H16O4 and C10H18O4 6.0 and 7.6 times). In the earlier segment, taken as O3, C10H16O4 and C10H18O4 rise 6.8 and 9.3 times, and the three nitrates also rise, by 1.18, 1.46 and 1.39 times, with one-sided p of 0.0002 to 0.001 on 5 against 9 points. The criterion is met only under r3; under r1 and r2 the nitrates also appear with O3, so the claim holds only in part. The README says nothing on which segment is which oxidant.

## Fig. 3B: wind tunnel

> fig3b-hyles-no3-eliminates: "In the wind tunnel, NO3 oxidation of the floral odor eliminates
> Hyles feeding visits; no moth fed at the NO3-oxidized odor or the no-odor control."
>
> fig3b-manduca-no3-halves: "For Manduca, NO3 oxidation lowers the feeding visitation rate by
> about 50%, to a level not different from the no-odor control."
>
> fig3b-o3-no2-no-effect: "O3 alone (120 ppb) or NO2 alone added to the floral odor does not
> change Manduca visitation relative to the untreated floral odor."
>
> fig3b-proxy-mixture: "The NO3-proxy mixture (84% less b-pinene, 67% less b-ocimene, no
> oxidation products) draws fewer Manduca feeding visits than the untreated floral odor, and as
> many as the clean-air control and the NO3-degraded odor."

The counts are in the Zenodo sheet only. Tests (criteria): (i) two-proportion z-test, pooled, no
continuity correction (Zar); (ii) R prop.test with Yates correction. The proxy mixture is the row
"blend of oxidized scent" (the only Manduca row left once the others are matched; an assumption on
a label, recorded).

In [ ]:
wt = chan.wind_tunnel_fig3b()["Sheet1"].iloc[:, :4]
wt["proportion"] = wt.Visits / wt["Moths tested"]
print(wt.round(3).to_string(index=False))
g = {(r.Species, r.Treatment): (r.Visits, r["Moths tested"]) for _, r in wt.iterrows()}
def cmp(sp, a, b):
    (x1, n1), (x2, n2) = g[(sp, a)], g[(sp, b)]
    return {"species": sp, "a": a, "b": b, "p_a": x1 / n1, "p_b": x2 / n2, **chan.proportion_tests(x1, n1, x2, n2)}
H, M = "H. lineata", "M. sexta"
pairs = [(H, "odor blend", "NO3 oxidized odor blend"), (H, "odor blend", "clean air control"),
         (M, "odor blend", "NO3 oxidized odor blend"), (M, "NO3 oxidized odor blend", "clean air control"),
         (M, "odor blend", "O3 oxidized odor blend"), (M, "odor blend", "NO2 oxidized odor blend"),
         (M, "odor blend", "blend of oxidized scent"), (M, "blend of oxidized scent", "clean air control"),
         (M, "blend of oxidized scent", "NO3 oxidized odor blend")]
tests = pd.DataFrame([cmp(*p) for p in pairs])
print(tests.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
mand = wt[wt.Species == M].reset_index(drop=True)
allp = []
for i, j in [(i, j) for i in range(len(mand)) for j in range(i + 1, len(mand))]:
    r = chan.proportion_tests(mand.Visits[j], mand["Moths tested"][j], mand.Visits[i], mand["Moths tested"][i])
    allp.append({"a": mand.Treatment[i], "b": mand.Treatment[j], "p (i)": r["p (i) z-test"], "p (ii)": r["p (ii) prop.test Yates"]})
allp = pd.DataFrame(allp); allp["p (i) Holm"] = chan.holm(allp["p (i)"]); allp["p (ii) Holm"] = chan.holm(allp["p (ii)"])
print("\nAll 15 Manduca pairs, with Holm adjustment over them:"); print(allp.to_string(index=False, float_format=lambda v: f"{v:.4g}"))

In [ ]:
pf, pn = g[(M, "odor blend")][0] / g[(M, "odor blend")][1], g[(M, "NO3 oxidized odor blend")][0] / g[(M, "NO3 oxidized odor blend")][1]
print(f"Manduca decrease: (a) relative {100 * (1 - pn / pf):.1f}% ; (b) difference {100 * (pf - pn):.1f} percentage points (expected 50 +- 5)")
hy = wt[wt.Species == H].set_index("Treatment")
print("Hyles feeding at NO3 odor:", hy.loc["NO3 oxidized odor blend", "Visits"], "of", hy.loc["NO3 oxidized odor blend", "Moths tested"],
      "| at clean air:", hy.loc["clean air control", "Visits"], "of", hy.loc["clean air control", "Moths tested"])

In [ ]:
# The Methods test Manduca at 1/100 floral equivalent in every Fig. 3B treatment ("treatments consisted of
# 1/100 floral equivalent of floral odor blend, ..."); the sheet's floral-odor row (11 of 12) has the counts of
# the 1x row of Fig. S7C (finding-fig3b-control-is-1x). Second reading of the floral reference: the 0.01x row
# of Fig. S7C. Both are run for every Manduca comparison with the floral odor.
s7r = s7c["Sheet1"].set_index("Treatment")
refs = {"Fig. 3B sheet (11 of 12)": g[(M, "odor blend")],
        "Fig. S7C 0.01x row (%d of %d)" % (s7r.loc["0.01x floral scent", "Visits"], s7r.loc["0.01x floral scent", "Moths tested"]):
            (int(s7r.loc["0.01x floral scent", "Visits"]), int(s7r.loc["0.01x floral scent", "Moths tested"]))}
rows = []
for ref, (xr, nr) in refs.items():
    for b in ["NO3 oxidized odor blend", "O3 oxidized odor blend", "NO2 oxidized odor blend", "blend of oxidized scent", "clean air control"]:
        xb, nb_ = g[(M, b)]
        rows.append({"floral reference": ref, "against": b, "p floral": xr / nr, "p other": xb / nb_,
                     "relative decrease %": 100 * (1 - (xb / nb_) / (xr / nr)), "difference pp": 100 * (xr / nr - xb / nb_),
                     **chan.proportion_tests(xr, nr, xb, nb_)})
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.4g}"))


**Result: fig3b-hyles-no3-eliminates is partially reproduced; fig3b-manduca-no3-halves is reproduced on the Fig. 3B sheet and not under the second floral reference, resting on the sheet; fig3b-o3-no2-no-effect and fig3b-proxy-mixture are reproduced under both references.** No Hyles fed at the NO3-oxidized odor (0 of 19) or the clean air (0 of 42), as the claim says, but only 5 of 35 fed at the floral odor, and floral odor against NO3-oxidized odor gives p = 0.084 by the z-test and 0.22 by prop.test, so the criterion's significance part fails. For Manduca on the sheet's floral-odor group (11 of 12), NO3 lowers feeding from 0.917 to 0.421, a relative decrease of 54.1% (49.6 percentage points), with p = 0.0058 (z-test) or 0.017 (prop.test), against the paper's 0.047, and NO3 against clean air gives p = 0.60. With the 0.01x row of Fig. S7C (10 of 14) as the floral reference, which the Methods' 1/100 floral equivalent points to, the decrease is 41.1% (29.3 points) with p = 0.094 (z-test) or 0.19 (prop.test), and the claim would fail. The sheet is the paper's own Fig. 3B data, so its reading is adopted, as an assumption. Neither floral reference gives the paper's p = 0.047. O3 (p = 0.066) and NO2 (p = 0.16) do not differ from the floral odor by the plain z-test, nor after Holm adjustment, nor with the 0.01x reference (p = 0.58 and 0.97). The proxy mixture is lower than the floral odor (p = 0.0026; 0.049 with the 0.01x reference) and does not differ from clean air (p = 0.83) or the NO3-oxidized odor (p = 0.74). The sheet holds 113 Manduca where the Methods give 138, and its floral-odor group of 12 is below the Methods' 15 to 42 moths per stimulus.

## Fig. S7: EAG sensitivity and dose response

> figs7-manduca-sensitivity: "Manduca antennae are more sensitive than Hyles antennae to the floral
> odor, and in the wind tunnel 1/100 floral equivalent is the lowest dose whose Manduca feeding
> does not differ from one floral equivalent (P = 0.09)."

EAG: mean of each species at each concentration of the file; readings (a) at 1e-6, 1e-4 and 1e-2
(Fig. S7A), (b) at most concentrations of the series. The duplicated Hyles series (finding) is kept
here and dropped as a variant. Behaviour: the Dryad dose-response sheet, tests (i) and (ii).

In [ ]:
raw, eag = chan.eag()
def means(e):
    return e.groupby(["conc", "species"]).mV.mean().unstack()
m_all = means(eag)
drop = eag[~((eag.species == "Hyles") & eag.row.between(21, 28))]
m_drop = means(drop)
for label, mm in [("all values", m_all), ("Hyles rows 21-28 dropped", m_drop)]:
    mm = mm.assign(**{"Manduca > Hyles": mm.Manduca > mm.Hyles})
    print(label); print(mm.round(2))
    a_ok = bool(mm.loc[[1e-6, 1e-4, 1e-2], "Manduca > Hyles"].all()); b_ok = mm["Manduca > Hyles"].mean() > 0.5
    print(f"  (a) Manduca larger at 1e-6, 1e-4, 1e-2: {a_ok}; (b) larger at {int(mm['Manduca > Hyles'].sum())} of {len(mm)} concentrations")
s7 = chan.wind_tunnel_s7c()["Sheet1"]
s7["proportion"] = s7.Visits / s7["Moths tested"]; print(s7.round(3).to_string(index=False))
ref = s7.iloc[0]
for _, r in s7.iloc[1:].iterrows():
    res = chan.proportion_tests(ref.Visits, ref["Moths tested"], r.Visits, r["Moths tested"])
    print(f"{r.Treatment} against {ref.Treatment}: " + ", ".join(f"{k} {v:.4g}" for k, v in res.items()))

**Result: reproduced under reading (i) of the behavioural test.** Mean Manduca EAG exceeds mean Hyles EAG at all nine concentrations (at 1e-6, 1e-4 and 1e-2: 13.1 against 10.0, 22.8 against 17.6, 42.8 against 22.7 mV), with or without the duplicated Hyles series. In the wind tunnel 0.01x does not differ from 1x (10 of 14 against 11 of 12, p = 0.19 by the z-test, 0.42 by prop.test; the paper gives 0.09), and the lower dose 0.001x does (13 of 22, p = 0.046 by the z-test; 0.11 by prop.test, which does not separate it).

## Fig. 3D and Tables S10, S11: field visits

> fig3d-field-no3-fewer-visits: "In the field, hawkmoth visits to the NO3-oxidized floral odor are
> fewer than to the untreated odor, and not different from the clean-air control."
>
> fig3d-real-and-synthetic-alike: "In the field, hawkmoth visitation to the synthetic floral
> odor, the live floral scent and live flowers does not differ."
>
> fig3d-visits-per-night: "An untreated flower is visited by a hawkmoth about twice a night (1.9),
> the oxidized scent 0.57 times a night, a drop of about 70%."

The Dryad Table_S10.xlsx holds the pooled counts and hours per treatment (no per-night counts).
Poisson GLM with log link (statsmodels), readings (a) counts rescaled to 49.5 h with no offset,
(b) raw counts with log(hours) as offset; reference floral odor, then the no-odor control.

In [ ]:
_, s10 = chan.table_s10()
counts = s10.loc["Total visits"].astype(float); hours = s10.loc["Hours analyzed"].astype(float)
def glm(ref, reading):
    trt = list(counts.index)
    order = [ref] + [t for t in trt if t != ref]
    X = pd.DataFrame({t: [1.0 if x == t else 0.0 for x in order] for t in order[1:]}, index=order)
    X = sm.add_constant(X)
    if reading == "a":
        y, off = counts[order] / hours[order] * 49.5, None
    else:
        y, off = counts[order], np.log(hours[order].to_numpy())
    f = sm.GLM(y.to_numpy(), X.to_numpy(), family=sm.families.Poisson(), offset=off).fit()
    return pd.DataFrame({"estimate": f.params, "se": f.bse, "z": f.tvalues, "p": f.pvalues}, index=["intercept"] + order[1:])
fits = {(ref, r): glm(ref, r) for ref in ["Floral odor", "No odor ctl."] for r in ["a", "b"]}
for k, v in fits.items():
    print("reference", k[0], "reading", k[1]); print(v.round(4))
print("Table S11: intercept 2.639 (0.267); no-odor -1.03 (0.521) p 0.048; NO3 -1.253 (0.567) p 0.027; real flower -0.027 (0.38) p 0.944; real scent 0.069 (0.372) p 0.853")

In [ ]:
for r in ["a", "b"]:
    f, c = fits[("Floral odor", r)], fits[("No odor ctl.", r)]
    no3 = f.loc["NO3 oxid. floral odor"]
    print(f"reading ({r}): NO3 coefficient {no3.estimate:.3f} (expected -1.253 +- 0.567), p {no3.p:.4f}; NO3 against control p {c.loc['NO3 oxid. floral odor', 'p']:.3f}; "
          f"live flower p {f.loc['Live flower', 'p']:.3f}, live scent p {f.loc['Live flower scent', 'p']:.3f}")
rate = counts / hours
print("visits per hour:", rate.round(3).to_dict(), "(expected floral odor 0.283 +- 0.076, NO3 0.081 +- 0.04)")

In [ ]:
night = rate * 7
for untreated in ["Live flower", "Floral odor"]:
    drop = 1 - night["NO3 oxid. floral odor"] / night[untreated]
    print(f"untreated = {untreated}: {night[untreated]:.3f} visits per night (1.9 +- 0.9); NO3 {night['NO3 oxid. floral odor']:.3f} (0.57 +- 0.28); drop {100 * drop:.1f}% (70 +- 20)")
print("Table S10 'Visits/7 hour night' row in the file:", s10.loc["Visits/7 hour night"].to_dict())

**Result: fig3d-field-no3-fewer-visits, fig3d-real-and-synthetic-alike and fig3d-visits-per-night are reproduced.** On the deposited Table S10 counts, the Poisson GLM gives an NO3 coefficient of -1.253 (SE 0.567, p = 0.027) under both readings, as Table S11; NO3 against the no-odor control gives p = 0.74. Live flower and live floral scent against the floral odor give p = 0.94 and 0.86 (reading a) or 0.93 and 0.86 (reading b). Visits per night are 1.928 for the live flower (1.980 for the floral odor) and 0.566 for the NO3-oxidized odor, a drop of 70.7% (71.4%). The deposit holds only pooled counts and hours per treatment, so these tests repeat the paper's computation on its table.

## Fig. 4, Fig. S8 and the GEOS-Chem fields

> figs8-no3-faster-than-o3: "At typical northern terrestrial levels (2.6 ppt NO3, 37 ppb O3) most
> floral compounds react faster with NO3 than with O3, the exceptions being the sesquiterpenes
> a-humulene and b-caryophyllene; monoterpenes are the most reactive."

Means over land cells (any land in the cell, global-land-mask 1.0.0, as the Supplementary Methods
say) with cell centres between 10 and 50 degrees north, July 2013 (month 7 of the 14), bottom
level of the 12.1.0 output in the Zenodo release. Rates are Table S12's constants times the
recomputed concentrations at Table S12's air density. Other averages are shown as variants.

In [ ]:
lat, lon = gc["lat"].ravel(), gc["lon"].ravel()
lf = chan.land_fraction(lat, lon); land = lf > 0
print("land cells:", int(land.sum()), "of", land.size)
NO3 = gc["Conc_NO3_Y0"][:, :, 0, :] * 1e12; O3 = gc["Conc_O3_Y0"][:, :, 0, :] * 1e9   # ppt, ppb
JUL, JAN = 6, 0
def band_mean(arr, month, lo, hi, mask=land):
    b = (lat >= lo) & (lat <= hi)
    return arr[:, b, month][mask[:, b]].mean()
no3_m, o3_m = band_mean(NO3, JUL, 10, 50), band_mean(O3, JUL, 10, 50)
print(f"July 2013, land, 10 to 50 N: NO3 {no3_m:.2f} ppt (expected 2.6 +- 0.26), O3 {o3_m:.1f} ppb (37 +- 3.7)")
var = []
for name, mo, lo, hi, mask in [("June, 10-50 N", 5, 10, 50, land), ("August, 10-50 N", 7, 10, 50, land),
                                ("July, 10-50 N, cells at least half land", JUL, 10, 50, lf >= 0.5),
                                ("July, 10-50 N, all cells", JUL, 10, 50, np.ones_like(land)),
                                ("July, 10-50 S", JUL, -50, -10, land), ("January, 10-50 S", JAN, -50, -10, land)]:
    var.append({"variant": name, "NO3 ppt": band_mean(NO3, mo, lo, hi, mask), "O3 ppb": band_mean(O3, mo, lo, hi, mask)})
b2 = (np.abs(lat) >= 10) & (np.abs(lat) <= 50)
for mo, nm in [(JUL, "July"), (5, "June")]:
    var.append({"variant": f"{nm}, 10-50 N and S", "NO3 ppt": NO3[:, b2, mo][land[:, b2]].mean(), "O3 ppb": O3[:, b2, mo][land[:, b2]].mean()})
print(pd.DataFrame(var).round(2).to_string(index=False))
k = pd.DataFrame(chan.TABLE_S12, columns=["compound", "class", "k_O3", "k_NO3"])
for label, n3, o3_ in [("recomputed", no3_m, o3_m), ("paper's 2.6 ppt and 37 ppb", 2.6, 37.0)]:
    k["O3 rate"] = k.k_O3 * chan.number_density(o3_ * 1e-9); k["NO3 rate"] = k.k_NO3 * chan.number_density(n3 * 1e-12)
    faster_o3 = list(k.compound[k["O3 rate"] > k["NO3 rate"]])
    top = k.loc[(k["O3 rate"] + k["NO3 rate"]).idxmax()]
    print(f"{label}: NO3 faster for {int((k['NO3 rate'] > k['O3 rate']).sum())} of 23; O3 faster for {faster_o3}; most reactive {top.compound} ({top['class']})")
# Weighted means over the same land cells (variants): by cell area, cos(latitude), and by the model's grid
# volume (Gridvolume_Y0, bottom level, July).
b10 = (lat >= 10) & (lat <= 50); m10 = land[:, b10]
for nm, w_ in [("area, cos(lat)", np.cos(np.deg2rad(lat))[None, :] * np.ones((len(lon), 1))), ("grid volume", gc["Gridvolume_Y0"][:, :, 0, JUL])]:
    print(f"July, 10-50 N, land, weighted by {nm}: NO3 {np.average(NO3[:, b10, JUL][m10], weights=w_[:, b10][m10]):.2f} ppt, "
          f"O3 {np.average(O3[:, b10, JUL][m10], weights=w_[:, b10][m10]):.1f} ppb")


**Result: partially reproduced.** The July 2013 bottom-level means over land cells between 10 and 50 N are 3.48 ppt NO3 and 43.0 ppb O3, outside 2.6 +- 0.26 and 37 +- 3.7; weighting by cell area (3.38 ppt, 42.4 ppb) or grid volume (3.39 ppt, 42.4 ppb) does not bring them within tolerance, and no variant over the northern band gives both. June over land in both hemispheres gives 2.69 ppt and 39.1 ppb, both within tolerance, but the criterion names the northern band only. The rate comparison, the finding itself, holds at both the recomputed and the paper's concentrations: NO3 is faster for 17 of 23 compounds, a-humulene and b-caryophyllene are faster with O3 (with a-terpinene and the three green leaf volatiles), and the most reactive compound is a-terpinene, a monoterpene. So the comparison holds and the stated concentrations do not.

## Recognition distances: Fig. 4, Table S13 and Fig. S9

The recognition distance is the degradation time times "the mean horizontal wind speed of the bottom vertical grid" (Supplementary
Methods, GEOS-Chem code availability). The deposited 12.1.0 output holds no wind. The model was driven by MERRA-2 2013 meteorology
(Supplementary Methods, Global models), which GEOS-Chem reads from its processed MERRA-2 files; the bottom-level U and V of those
files for July and January 2013 are read here from the public GEOS-Chem data bucket at 0.5 x 0.625 degrees and averaged to the
2 x 2.5 degree grid (`chan.merra2_bottom_wind`, `chan.to_gc_grid`). This is an assumption: the paper's own wind field is not
deposited, and the 2 x 2.5 degree files for 2013 are not on the bucket. City coordinates and populations are from the UN World
Urbanization Prospects 2018 (`chan.table_s13_cities`), since the source Table S13 cites (ref. 79, World Population Prospects 2019
Highlights) gives no coordinates; this too is an assumption.

Readings, as the criteria name them: density (1) at the cell's T and P or (2) Table S12's; threshold (a) either compound or (b)
both; NO3 map (i) NO3 alone or (ii) NO3 and O3; wind (x) speed of the monthly mean vector or (y) monthly mean speed; city (containing
cell, or bilinear interpolation of the map); factor map regridded by nearest cell or bilinearly. Summer months: July north of the
equator, January south of it; land cells (any land) south of -56 degrees removed.


In [ ]:
wind_raw = {mo: chan.merra2_bottom_wind(mo) for mo in (7, 1)}
for mo, w_ in wind_raw.items():
    print(f"MERRA-2 2013-{mo:02d}: {w_['n_days']} days, {w_['n_fields']} 3-hourly fields; level-1 speed {w_['speed'].min():.2f} to {w_['speed'].max():.2f} m/s, "
          f"U {w_['u'].min():.2f} to {w_['u'].max():.2f}, V {w_['v'].min():.2f} to {w_['v'].max():.2f}")
month = np.where(lat >= 0, JUL, JAN)
def summer(arr):
    return np.stack([arr[:, j, month[j]] for j in range(len(lat))], axis=1)
wgrid = {}
for mo, key in [(7, JUL), (1, JAN)]:
    gg = {k: chan.to_gc_grid(wind_raw[mo][k], lat, lon) for k in ("u", "v", "speed")}
    wgrid[key] = {"(x) vector": np.hypot(gg["u"], gg["v"]), "(y) speed": gg["speed"]}
WIND = {r: np.stack([wgrid[month[j]][r][:, j] for j in range(len(lat))], axis=1) for r in ["(x) vector", "(y) speed"]}
keep = land & (lat[None, :] >= -56)
print("land cells kept:", int(keep.sum()))
for r, a_ in WIND.items():
    print(f"summer bottom-level wind {r} over kept land cells: median {np.median(a_[keep]):.2f} m/s, range {a_[keep].min():.2f} to {a_[keep].max():.2f}")


In [ ]:
import itertools
from scipy.interpolate import RegularGridInterpolator
no3_vv, o3_vv = summer(gc["Conc_NO3_Y0"][:, :, 0, :]), summer(gc["Conc_O3_Y0"][:, :, 0, :])
Ps, Ts = summer(gc["Pres_Y0"][:, :, 0, :]), summer(gc["Temp_Y0"][:, :, 0, :])
kb = {"b-pinene": (1.50e-17, 2.51e-12, 0.84), "ocimene": (3.74e-16, 2.20e-11, 0.67)}   # Table S12 k(O3), k(NO3); Fig. 4 thresholds
DENS = {"(1) cell T, P": lambda x: chan.number_density(x, Ps, Ts), "(2) Table S12": chan.number_density}
THR = {"(a) either": np.minimum, "(b) both": np.maximum}
def deg_time(no3, o3, dens, thr, nmap):
    """Seconds to the threshold loss under first-order loss: O3 alone, NO3 alone (i), NO3 and O3 (ii)."""
    ts = []
    for ko, kn, f in kb.values():
        rate = {"O3": ko * dens(o3), "(i)": kn * dens(no3), "(ii)": kn * dens(no3) + ko * dens(o3)}[nmap]
        ts.append(-np.log(1 - f) / rate)
    return THR[thr](*ts)
dist = {(d, t_, wr, nm): deg_time(no3_vv, o3_vv, DENS[d], t_, nm) * WIND[wr]
        for d, t_, wr, nm in itertools.product(DENS, THR, WIND, ["O3", "(i)", "(ii)"])}
# Factor maps (13.2.1 preindustrial over 2013): dimensions, order and values over Greenland.
fm = {s: chan.factor_map(s) for s in ["NO3", "O3"]}
import h5py, io
with h5py.File(io.BytesIO(chan.zread("Figure 4/GEOS-Chem 13.2.1 NO3_pi_pd_factor_map 2013.nc4")), "r") as h_:
    dimnames = {h_[k].attrs["_Netcdf4Dimid"]: k for k in ("time", "lat", "lon")}
    print("SpeciesConc_NO3 shape", h_["SpeciesConc_NO3"].shape, "dimensions", [dimnames[i] for i in h_["SpeciesConc_NO3"].attrs["_Netcdf4Coordinates"]])
f_ = fm["NO3"]; print("factor-map lat", f_["lat"][:3], "...", f_["lat"][-2:], "| lon", f_["lon"][:3], "...", f_["lon"][-2:])
gl_, go_ = (f_["lat"] >= 60) & (f_["lat"] <= 84), (f_["lon"] >= -75) & (f_["lon"] <= -10)
print("July NO3 factor over 60-84 N, 75-10 W (rows lat, columns lon):")
print(pd.DataFrame(f_["factor"][JUL][np.ix_(gl_, go_)], index=f_["lat"][gl_], columns=f_["lon"][go_]).round(2).to_string())
glon, glat = np.meshgrid(lon, lat, indexing="ij")
def regrid(s, mo, method):
    f = fm[s]; lonx = np.concatenate([f["lon"], [180.0]]); fac = f["factor"][mo]
    fac = np.concatenate([fac, fac[:, :1]], axis=1)
    return RegularGridInterpolator((f["lat"], lonx), fac, method=method, bounds_error=False, fill_value=None)((glat, glon))
pct = {}
for method in ["nearest", "linear"]:
    fac = {s: np.stack([regrid(s, month[j], method)[:, j] for j in range(len(lat))], axis=1) for s in ["NO3", "O3"]}
    for t_, nm in itertools.product(THR, ["(i)", "(ii)"]):
        dens = DENS["(2) Table S12"]   # the ratio does not depend on the density or the wind
        pct[(method, t_, nm)] = 100 * deg_time(no3_vv, o3_vv, dens, t_, nm) / deg_time(no3_vv * fac["NO3"], o3_vv * fac["O3"], dens, t_, nm)
for k, v in dist.items():
    assert np.isfinite(v[keep]).all(), k
print("distance maps:", len(dist), "| percentage maps:", len(pct))
print("present distance over kept land cells, range over all maps:", f"{min(v[keep].min() for v in dist.values()):.3g} to {max(v[keep].max() for v in dist.values()):.3g} m")


In [ ]:
cities = chan.table_s13_cities()
print(len(cities), "Table S13 cities matched in WUP 2018; population 2018", f"{cities.population_2018.min():.0f} to {cities.population_2018.max():.0f}")
lonx = np.concatenate([lon, [180.0]])
def at_cities(m, how):
    if how == "containing cell":
        i = np.array([np.argmin(np.abs((lon - x + 180) % 360 - 180)) for x in cities.lon])
        j = np.array([np.argmin(np.abs(lat - y)) for y in cities.lat])
        return m[i, j]
    return RegularGridInterpolator((lonx, lat), np.concatenate([m, m[:1]], axis=0), method="linear")(np.c_[cities.lon, cities.lat])
EXP_D = {"Kabul": 73.1, "Santiago": 101.8, "Milan": 157.6, "Barcelona": 177.4, "Los Angeles": 452.4}
EXP_P = {"Hong Kong": 4.9, "Tokyo": 6.7, "Kinshasa": 89.8}
HOW = ["containing cell", "interpolated"]
rows, city_d, city_p = [], {}, {}
for (d, t_, wr, nm), m_ in dist.items():
    if nm == "O3":
        continue
    for how in HOW:
        dc = pd.Series(at_cities(m_, how), index=cities.city); city_d[(d, t_, wr, nm, how)] = dc
        for method in ["nearest", "linear"]:
            pc = pd.Series(at_cities(pct[(method, t_, nm)], how), index=cities.city); city_p[(method, t_, nm, how)] = pc
            vals = [dc[c] for c in EXP_D] + [pc[c] for c in EXP_P]
            n_ok = sum(abs(v - e) <= 0.1 * e for v, e in zip(vals, list(EXP_D.values()) + list(EXP_P.values())))
            rows.append({"density": d, "threshold": t_, "wind": wr, "NO3 map": nm, "city": how, "factor": method,
                         **{c: v for c, v in zip(list(EXP_D) + [c + " %" for c in EXP_P], vals)}, "within tol (of 8)": n_ok,
                         "four shortest": ", ".join(dc.nsmallest(4).index)})
s13 = pd.DataFrame(rows)
print("Expected (m):", EXP_D, "| expected (%):", EXP_P)
print(s13.to_string(index=False, float_format=lambda v: f"{v:.1f}"))
print("most values within tolerance under any reading:", s13["within tol (of 8)"].max(), "of 8 (6 needed for partial, 8 for reproduced)")


**Result: table-s13-recognition-distances is not reproduced.** Under no reading are more than 1 of the 8 values within tolerance (6 are needed for partial reproduction). With NO3 and O3, the speed of the mean wind vector and the containing cell, Kabul is 105 to 135 m against 73.1, Santiago 512 to 634 m against 101.8, Milan 176 to 198 m against 157.6, Barcelona 264 to 281 m against 177.4 and Los Angeles 298 to 326 m against 452.4; Hong Kong and Tokyo are at 1.8% and 2.2 to 2.4% of their preindustrial distance against 4.9 and 6.7%, and Kinshasa at 70.7 to 73.3% against 89.8%. The monthly mean speed (y) gives distances several times longer, and the threshold reading (b) about 15 times longer. The four shortest are Kabul, Milan, London and Delhi or Madrid under the vector-wind readings, against Kabul, Santiago, Milan and Barcelona. This rests on the MERRA-2 wind and the WUP 2018 coordinates, both assumed. The three percentages need no wind and fail under every reading, so even if all five distances matched, only 5 of the 8 values could pass, short of the 6 needed for partial reproduction.

In [ ]:
rows = []
for d, t_, wr in itertools.product(DENS, THR, WIND):
    o3m = dist[(d, t_, wr, "O3")]
    for nm in ["(i)", "(ii)"]:
        nm_ = dist[(d, t_, wr, nm)]
        for how in HOW:
            dc = city_d[(d, t_, wr, nm, how)]
            n400 = int((dc < 400).sum())
            share = (nm_ < o3m)[keep].mean()
            rows.append({"density": d, "threshold": t_, "wind": wr, "NO3 map": nm, "city": how,
                         "share land cells NO3 shorter": share, "median O3 map (m)": np.median(o3m[keep]), "median NO3 map (m)": np.median(nm_[keep]),
                         "cities < 400 m": n400, "more severe": share > 0.5, "8 to 10 below 400 m": 8 <= n400 <= 10,
                         "both hold": share > 0.5 and 8 <= n400 <= 10, "cities below 400 m": ", ".join(dc[dc < 400].sort_values().index)})
f4ab = pd.DataFrame(rows)
print(f4ab.to_string(index=False, float_format=lambda v: f"{v:.3g}"))
print("readings where both hold:", int(f4ab["both hold"].sum()), "of", len(f4ab))


**Result: fig4ab-no3-shortens-recognition is partially reproduced, resting on the assumed wind and city coordinates: the "more severe" part holds and the below 400 m part is not matched.** The NO3 map is shorter than the O3 map in all land cells when NO3 and O3 act together (ii), under every density, threshold and wind reading, and in 37% (a) or 51% (b) of them with NO3 alone (i); this part needs no wind. The number of Table S13 cities below 400 m is within one of the paper's nine only with the speed of the mean wind vector (x) and threshold (a): 8 (density 1, containing cell, NO3 alone), 10 (density 1, containing cell, NO3 and O3), 9 (density 2, containing cell, NO3 alone) and 8 (density 2, interpolated, NO3 and O3). Both parts hold under the two with NO3 and O3. With the monthly mean speed (y) the count is 0 or 1.

The cell below gives the analysis behind the outcome. Under the two readings where both parts hold, the same readings match 0 and 1 of the 8 Table S13 values, and 6 of the 10 and 5 of the 8 cities below 400 m are among the printed table's nine (Barcelona, Jiddah, Kabul, Lahore, Madrid, Milan, Paris, Riyadh, Santiago); London, Delhi, Los Angeles, Rome and Beijing are on the recomputed lists and not in the paper's. The count of 8 or 10 therefore matches the paper's nine in number only, not in the cities it names, and the dependency, table-s13-recognition-distances, is not matched under those readings. The criterion ties "many populated areas" to that claim, so the 400 m part is not matched, and the outcome rests on this, not on which wind reading is preferred: the paper worked from monthly-averaged output, and both the monthly mean speed (y) and the speed of the mean wind vector (x) are plausible readings of its "mean horizontal wind speed". The vector speed falls to 0.05 m/s over land, where wind directions cancel, and is below 0.5 m/s in 115 of 4197 cells. The criterion defines no partial outcome; partially reproduced follows its reference to table-s13-recognition-distances. The outcome was reproduced (assumed) until the third verification round, which agreed with this change.

In [ ]:
# Proposed change: for each reading, the cities below 400 m beside how many of the 8 Table S13 values
# the same reading matches (best of the two factor maps), and how many of those cities are among the
# nine below 400 m in the printed Table S13.
s13p = chan.table_s13_printed()
nine = set(s13p.loc[s13p.distance_m < 400, "city"])
print("printed Table S13, cities below 400 m:", len(nine), sorted(nine))
key = ["density", "threshold", "wind", "NO3 map", "city"]
m4 = f4ab.merge(s13.groupby(key)["within tol (of 8)"].max().rename("S13 within tol (of 8)").reset_index(), on=key)
m4["of the printed nine"] = [len(nine & set(c.split(", "))) if c else 0 for c in m4["cities below 400 m"]]
print(m4[m4["8 to 10 below 400 m"]][key + ["more severe", "cities < 400 m", "of the printed nine", "S13 within tol (of 8)", "both hold"]].to_string(index=False))
for wr, g in m4.groupby("wind"):
    print(f"{wr}: cities below 400 m {g['cities < 400 m'].min()} to {g['cities < 400 m'].max()}; more severe in {int(g['more severe'].sum())} of {len(g)}; both hold in {int(g['both hold'].sum())}")
print("more severe, by NO3 map:", m4.groupby("NO3 map")["share land cells NO3 shorter"].agg(["min", "max"]).to_dict("index"))
vx = WIND["(x) vector"][keep]
print(f"(x) vector speed over kept land cells: {vx.min():.2f} m/s minimum, {int((vx < 0.5).sum())} of {vx.size} cells below 0.5 m/s; (y) minimum {WIND['(y) speed'][keep].min():.2f} m/s")


In [ ]:
rows = []
pop = cities.population_2018.to_numpy()
for (method, t_, nm, how), pc in city_p.items():
    pm = pct[(method, t_, nm)]
    green = keep & (glat >= 60) & (glon >= -73) & (glon <= -12)
    rows.append({"factor": method, "threshold": t_, "NO3 map": nm, "city": how,
                 "(a) share of cities <= 25%": (pc <= 25).mean(), "(b) share of population <= 25%": pop[(pc <= 25).values].sum() / pop.sum(),
                 "median % over land": np.median(pm[keep]), "Greenland cells": int(green.sum()), "Greenland median %": np.median(pm[green]),
                 "Greenland share > 75%": (pm[green] > 75).mean()})
f4c = pd.DataFrame(rows)
f4c["majority"] = (f4c["(a) share of cities <= 25%"] > 0.5) | (f4c["(b) share of population <= 25%"] > 0.5)
f4c["Greenland above 75%"] = f4c["Greenland share > 75%"] == 1
print(f4c.to_string(index=False, float_format=lambda v: f"{v:.3g}"))
print("Greenland is taken as land cells at 60 N or more between 73 W and 12 W (a standard box, not the paper's).")
pm = pct[("linear", "(a) either", "(ii)")]
gi, gj = (lon >= -73) & (lon <= -12), lat >= 60
print("present over preindustrial distance (%), factor linear, (a), NO3 and O3, over the Greenland box (land cells; others blank):")
print(pd.DataFrame(np.where(keep, pm, np.nan)[np.ix_(gi, gj)].T, index=lat[gj], columns=lon[gi]).round(0).to_string(na_rep=""))


**Result: fig4c-preindustrial-change is not reproduced.** Most Table S13 cities are at 25% or less of their preindustrial distance under every reading (68% to 90% of the cities, 73% to 92% of their population), so the first part holds. The second does not: over the Greenland box the median is 34% to 66% and at most 0.4% of land cells are above 75%, under every reading, so the change there is not small. The partial outcome needs the Table S13 percentages matched and no majority, and neither holds. The Greenland part needs no wind or city coordinates.

In [ ]:
# Fig. S9 on the rows Table S13 prints (97 cities: population, present distance, percentage of
# preindustrial). These need no wind and no city coordinates.
import statsmodels.api as sm
from scipy.stats import spearmanr
def quad(x, y):
    return sm.OLS(y, sm.add_constant(np.c_[x, x ** 2])).fit()
s13p = chan.table_s13_printed()
print(len(s13p), "rows; ranks", s13p["rank"].min(), "to", s13p["rank"].max(), "| shortest:", s13p.nsmallest(4, "distance_m")[["city", "distance_m"]].values.tolist())
rows = []
for popr in ["raw", "log"]:
    px = np.log10(s13p.population.to_numpy(float)) if popr == "log" else s13p.population.to_numpy(float) / 1e6
    d_, p_ = s13p.distance_m.to_numpy(), s13p.percent_of_preindustrial.to_numpy()
    for name, (x, y) in {"A distance on population": (px, d_), "B impact on population": (px, p_),
                         "C impact on distance": (d_, p_)}.items():
        f = quad(x, y)
        rows.append({"population": popr, "regression": name, "R2": f.rsquared, "adj R2": f.rsquared_adj, "F": f.fvalue,
                     "df": f"{int(f.df_model)},{int(f.df_resid)}", "p": f.f_pvalue})
s9p = pd.DataFrame(rows); print(s9p.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
print("paper: A R2 0.008, F2,96 0.39, p 0.67; B R2 0.09, F2,96 4.89, p < 0.01; C R2 0.41, F2,96 32.78, p < 0.001")
rho = spearmanr(s13p.distance_m, s13p.percent_of_preindustrial)
print(f"Spearman rho (distance, % of preindustrial): {rho[0]:.3f}, p {rho[1]:.2g} (positive: shorter distance, larger impact)")
for popr, g in s9p.groupby("population", sort=False):
    a_, b_, c_ = g.set_index("regression").loc[["A distance on population", "B impact on population", "C impact on distance"]].to_dict("records")
    met = a_["p"] >= 0.05 and b_["p"] < 0.01 and c_["p"] < 0.001 and min(abs(c_["R2"] - 0.41), abs(c_["adj R2"] - 0.41)) <= 0.041 and rho[0] > 0
    print(f"population {popr}: criterion met {met}")


In [ ]:
# Variant: the same regressions on the recomputed distances and percentages, under every reading.
rows = []
for (d, t_, wr, nm, how), dc in city_d.items():
    for method in ["nearest", "linear"]:
        pc = city_p[(method, t_, nm, how)].to_numpy()
        for popr in ["raw", "log"]:
            px = np.log10(pop) if popr == "log" else pop / 1e6
            A, B, Cr = quad(px, dc.to_numpy()), quad(px, pc), quad(dc.to_numpy(), pc)
            rows.append({"density": d, "threshold": t_, "wind": wr, "NO3 map": nm, "city": how, "factor": method, "population": popr,
                         "A p": A.f_pvalue, "B p": B.f_pvalue, "B R2": B.rsquared, "C p": Cr.f_pvalue, "C R2": Cr.rsquared, "C adj R2": Cr.rsquared_adj,
                         "rho(distance, %)": spearmanr(dc, pc)[0]})
s9 = pd.DataFrame(rows)
r2ok = (abs(s9["C R2"] - 0.41) <= 0.041) | (abs(s9["C adj R2"] - 0.41) <= 0.041)
s9["met"] = (s9["A p"] >= 0.05) & (s9["B p"] < 0.01) & (s9["C p"] < 0.001) & r2ok & (s9["rho(distance, %)"] > 0)
print(s9.to_string(index=False, float_format=lambda v: f"{v:.3g}"))
print(f"readings: {len(s9)}; A p >= 0.05 in {int((s9['A p'] >= 0.05).sum())}; B p < 0.01 in {int((s9['B p'] < 0.01).sum())}; "
      f"C p < 0.001 in {int((s9['C p'] < 0.001).sum())}; C R2 (plain or adjusted) within 0.41 +- 0.041 in {int(r2ok.sum())}; all met in {int(s9.met.sum())}")
print(s9[s9.met].to_string(index=False, float_format=lambda v: f"{v:.3g}"))


**Result: figs9-urban-regressions is reproduced on the rows Table S13 prints.** On the 97 printed rows, with population raw (in millions) or log10, distance is not related to population (A: R2 0.008, F 0.39, p 0.68 raw; p 0.47 log), the impact is weakly related to population (B: R2 0.094, F 4.90, p 0.0095 raw; R2 0.096, p 0.0085 log) and to distance (C: R2 0.411, adjusted 0.398, F 32.79, p 1.6e-11), with shorter distances going with a larger impact (Spearman rho 0.62). These match Fig. S9's R2 0.008, 0.09 and 0.41 and F of 0.39, 4.89 and 32.78; the degrees of freedom are 2 and 94 for 97 rows, where the paper gives 2 and 96. This test needs no wind and no city coordinates. On the recomputed distances and percentages (variant, cell above) all parts hold in 4 of the 128 readings, all with the monthly mean speed (y), and the distances behind them do not match Table S13 (table-s13-recognition-distances).

## Table S1: field conditions, midnight to 6 am

> table-s1-field-conditions: "The field site is near-pristine; from midnight to 6 am, O3 averaged
> 37.0 ppb, NO2 0.72 ppb and horizontal wind 1.14 m/s."

Means of all samples from midnight to 6 am, and of nightly means (the criterion's two readings).
NO2: rows with a finite value; status 10004 only as a variant. Wind: the horizontal speed
sqrt(u^2 + v^2); the file's rows cannot be placed in time (finding), so the readings are all rows,
and the rows that would fall between midnight and 6 am if the log ran without a break from its
header time. The paper's "under stable conditions" is not defined and is not applied.

In [ ]:
o3n = o3[chan.night_mask(o3.time)]
cn = caps[chan.night_mask(caps.time) & caps.NO2.notna()]
cn_ok = cn[cn.Status == 10004]
ibn = ib[chan.night_mask(ib.time)]
def two(series, times):
    return series.mean(), series.groupby((times - pd.Timedelta("6h")).dt.date).mean().mean()
rows = []
for name, (s, t_), (exp, tol) in [("O3 ppb", (o3n.O3, o3n.time), (37.0, 4.1)), ("NO2 ppb", (cn.NO2, cn.time), (0.72, 0.28)),
                                  ("NO2 ppb, status 10004", (cn_ok.NO2, cn_ok.time), (0.72, 0.28)), ("RH %", (ibn.RH, ibn.time), (42.6, 5.0))]:
    a_, b_ = two(s, t_)
    rows.append({"variable": name, "n": len(s), "nights": t_.dt.date.nunique(), "mean of samples": a_, "mean of nightly means": b_,
                 "expected": exp, "tol": tol, "samples within": abs(a_ - exp) <= tol, "nightly within": abs(b_ - exp) <= tol})
hs = np.hypot(anem.u.to_numpy(), anem.v.to_numpy()) / 100
start = pd.Timestamp("2018-08-01 22:15:22")
i0, i1 = int((pd.Timestamp("2018-08-02 00:00") - start).total_seconds() * 32), int((pd.Timestamp("2018-08-02 06:00") - start).total_seconds() * 32)
for name, sel in [("wind m/s, all rows", slice(None)), ("wind m/s, rows 0-6 am if continuous", slice(i0, i1))]:
    v = hs[sel]
    rows.append({"variable": name, "n": len(v), "nights": 1, "mean of samples": v.mean(), "mean of nightly means": v.mean(),
                 "expected": 1.14, "tol": 0.48, "samples within": abs(v.mean() - 1.14) <= 0.48, "nightly within": abs(v.mean() - 1.14) <= 0.48})
tab = pd.DataFrame(rows); print(tab.round(3).to_string(index=False))
print("speed of the mean horizontal wind vector (all rows):", round(float(np.hypot(anem.u.mean(), anem.v.mean()) / 100), 3), "m/s")
print("night temperature from the iButtons:", round(ibn.temp_C.mean(), 2), "C (Table S1: 20.65)")
print("per-night means:"); print(o3n.groupby(o3n.time.dt.date).O3.mean().round(2).to_dict(), cn.groupby(cn.time.dt.date).NO2.mean().round(3).to_dict(), ibn.groupby(ibn.time.dt.date).RH.mean().round(1).to_dict())
# Clock of the ozone and CAPS logs. The README gives the ozone time as "time (24h clock)" and the CAPS
# Timestamp with no zone; only the iButton files state a zone ("Timezone",-7, line 2). Hours the logs cover
# as written, with the mean per hour, and the midnight to 6 am means if the logs were in UTC (local = UTC - 7).
print("README:", [l[:160] for l in chan.dryad("README.md").read_text().splitlines() if any(k in l for k in ("24h clock", "IgorTime", "ibutton format"))])
print("iButton line 2:", [chan.dryad(f).read_text(encoding="latin-1").splitlines()[1] for f in sorted(ib.file.unique())])
c_ok = caps[caps.NO2.notna()]
print("O3 per hour as logged (n, mean):", {h: (len(d), round(d.O3.mean(), 1)) for h, d in o3.groupby(o3.time.dt.hour)})
print("NO2 per hour as logged (n, mean):", {h: (len(d), round(d.NO2.mean(), 2)) for h, d in c_ok.groupby(c_ok.time.dt.hour)})
for shift, label in [(0, "as logged (local)"), (-7, "if logged in UTC")]:
    out = []
    for nm, df, col in [("O3", o3, "O3"), ("NO2", c_ok, "NO2")]:
        tl = df.time + pd.Timedelta(hours=shift); sel = chan.night_mask(tl)
        out.append(f"{nm} n = {int(sel.sum())}, mean {df[col][sel].mean():.3f}, nightly {df[col][sel].groupby((tl[sel] - pd.Timedelta('6h')).dt.date).mean().mean():.3f}")
    print(f"midnight to 6 am, {label}: " + "; ".join(out))


**Result: partially reproduced.** From midnight to 6 am, O3 averages 34.5 ppb (34.2 over nightly means; 37.0 +- 4.1), NO2 0.995 ppb (0.994; 0.72 +- 0.28) and relative humidity 47.7% (47.5% over nightly means; 42.6 +- 5.0), all three within tolerance under the nightly-mean reading, NO2 and RH near the edge. The horizontal wind speed is 2.11 m/s over all anemometer rows and 1.70 m/s over the rows that would fall between midnight and 6 am if the log were continuous, both outside 1.14 +- 0.48. The anemometer rows cannot be placed in time, and the paper's selection of stable conditions is not defined, so the wind comparison rests on rows the paper may not have used. The ozone and CAPS clocks are taken as local time: the ozone logs cover 23:00 to 07:59 as written, and read as UTC they would hold only the hour after local midnight in the window (O3 28.3 ppb, outside tolerance). All field files are from 2018, where Table S1 averages 2018 and 2019.

## Blocked claims

- fig1d-assemblage-day-night: the video log of visits (Table S2) is in neither the Dryad deposit
  nor the Zenodo release (data missing).
- fig2a-gcead-monoterpenes: the claim's test needs the 8 Manduca and 13 Hyles antennae; the
  deposit holds one GC-EAD trace per moth species (and one Megachile), with no peak
  identifications or retention times to place b-pinene and cis-b-ocimene (data missing). Placing
  the peaks of the one trace by the Fig. S2 labels would need the figure's retention times read by
  eye, and one antenna per species cannot show a response in most antennae, so it was not done.
- table-s6-emission-rates: the 35 headspace samples and the synthetic standards that convert peak
  areas to ng/h are not deposited; Zenodo holds the raw GC-MS run of one field sample and Dryad the
  TIC of the synthetic blend (data missing). The quantitation files beside the field sample, listed
  below, hold a one-level calibration of 1,4-dimethoxybenzene only.
- fig2-dose-dependent: no flowtube run at another NO3 dose or exposure time is deposited (data
  missing).
- interannual-similar: the 13.2.1 runs for 2013, 2019 and 2021 are not deposited; only the
  preindustrial over 2013 factor maps are (data missing).


In [ ]:
# table-s6: the quantitation files beside the one field GC-MS run in the Zenodo release.
import re
p_ = "Fig S1 Table S5 raw GCMS folder Oenothera pallida floral scent/"
print(chan.zread(p_ + "fileinfo.txt").decode("latin-1").strip())
q = pd.read_excel(io.BytesIO(chan.zread(p_ + "MARIE_FLOWERSAMPLES_110802.XLS")), sheet_name=None, header=None, engine="xlrd")
for k, v in q.items():
    v = v.dropna(how="all")
    print(f"\nMARIE_FLOWERSAMPLES_110802.XLS [{k}], non-empty rows:"); print(v.iloc[:, :9].to_string(header=False, na_rep=""))
res = chan.zread(p_ + "MARIE_FLOWERSAMPLES_110802.RES")
print("\ntext in MARIE_FLOWERSAMPLES_110802.RES:", sorted({s.decode("utf-16-le") for s in re.findall(rb"(?:[\x20-\x7e]\x00){4,}", res)}))


In [ ]:
print("runtime %.1f min, peak memory %.2f GB" % ((time.time() - T0) / 60, resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6))

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| fig1d-assemblage-day-night | blocked (data missing) |
| fig1e-bagged-fewer-fruit | reproduced |
| fig1e-nocturnal-exclusion | reproduced |
| fig1e-groups | reproduced |
| nocturnal-share-of-fruit | reproduced |
| fig2a-gcead-monoterpenes | blocked (data missing) |
| table-s6-emission-rates | blocked (data missing) |
| fig2d-no3-degrades-monoterpenes | reproduced |
| fig2d-other-compounds-unchanged | reproduced |
| figs5-blend-oxidation | partially reproduced |
| fig2-dose-dependent | blocked (data missing) |
| figs6-oxidation-products | partially reproduced (assumed) |
| fig3b-hyles-no3-eliminates | partially reproduced |
| fig3b-manduca-no3-halves | reproduced (assumed) |
| fig3b-o3-no2-no-effect | reproduced |
| fig3b-proxy-mixture | reproduced (assumed) |
| figs7-manduca-sensitivity | reproduced |
| fig3d-field-no3-fewer-visits | reproduced |
| fig3d-real-and-synthetic-alike | reproduced |
| fig3d-visits-per-night | reproduced |
| fig4ab-no3-shortens-recognition | partially reproduced (assumed) |
| table-s13-recognition-distances | not reproduced (assumed) |
| fig4c-preindustrial-change | not reproduced (assumed) |
| interannual-similar | blocked (data missing) |
| figs8-no3-faster-than-o3 | partially reproduced |
| figs9-urban-regressions | reproduced |
| table-s1-field-conditions | partially reproduced |

How the analysis differs from the paper's. The fruit-set counts, the Fig. 3B counts, the raw Vocus traces and the GEOS-Chem fields are not in the Dryad deposit and were read from the paper's Zenodo code release. The flowtube stage windows of Fig. 2D and Table S9 are those of the paper's own notebook in the Dryad deposit, as the claims file now says after a reader's answer, with the last half and the whole of each stage as variants; the blend run of Fig. S5 uses the last half and the whole of each stage; the blend CIMS signals are not divided by the reagent ion, which is not deposited. The Fig. 3B floral-odor group is the sheet's (11 of 12), with the 0.01x row of Fig. S7C as a second reading. The field tests use the pooled Table S10 counts, as the paper does, and the fruit-set tests the counts of the paper's R script: the deposit holds no flower-level or night-level data. The recognition distances use the bottom-level MERRA-2 wind of July and January 2013 from GEOS-Chem's processed 0.5 x 0.625 degree files, averaged to the 2 x 2.5 degree grid, as the model's own wind is not deposited, and the city coordinates and populations of the UN World Urbanization Prospects 2018, as the source Table S13 cites gives no coordinates; under these the Table S13 distances are not matched. The Fig. S9 regressions are run on the rows Table S13 prints, and on the recomputed values as a variant.

What was found about the dataset. The Dryad deposit holds the field logs, the flowtube CIMS and Fig. 2D traces, the EAG and dose-response sheets, the Fig. S6 products and Table S10, but none of the GEOS-Chem output its description is about, no pollinator or fruit-set data, and no Fig. 3B data. In the EAG file one Hyles dose series repeats eight values of another, and the file holds 159 values per species (about 18 series of 9 concentrations), not 159 moths. The Manduca floral-odor group of Fig. 3B has the counts of the 1x dose-response row, where the Methods give 1/100 floral equivalent, and the Fig. 3B sheet holds 113 Manduca where the Methods give 138. The deposited Fig. 2D b-pinene trace gives 29% and 74% losses, where the raw trace gives Table S9's 33% and 84%; its time column is in seconds, not the README's minutes. The anemometer file has no timestamps, and its wind speeds (1.7 to 2.1 m/s) are above Table S1's 1.14 m/s. The CAPS files carry an IgorTime column that is not the clock and undocumented status codes, and neither the ozone nor the CAPS files give a time zone. The Fig. S6 workbook does not say which segment is which oxidant.
